In [ ]:
# Imports
import os
import time
import re
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup, Comment


In [ ]:
# Basic settings I might want to change later
CBB_BASE_URL = "https://www.sports-reference.com/cbb/players"
# File with NBA players and names
PLAYER_IDS_CSV = "player_ids.csv"
# Where college data will be saved
COLLEGE_STATS_CSV = "college_players_stats.csv"
# Only keep college seasons from 2010 and newer (Advanced stats weren't being tracked before this time)
START_YEAR = 2010


In [ ]:
def get_driver():
    # Make the browser settings for Selenium
    options = Options()
   # Path to Brave
    options.binary_location = r"C:\Program Files\BraveSoftware\Brave-Browser\Application\brave.exe"

    # Run without showing the browser window
    options.add_argument('--headless')

    # These help avoid common Brave issues
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu')

    # Set the browser window size so pages load consistently
    options.add_argument('window-size=1920,1080')

    # Pretend to be a normal user
    options.add_argument(
        'user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36'
    )

    # This makes sure Selenium uses the right driver version for Brave
    service = Service(ChromeDriverManager().install())

    # Start up the browser with the options I just set
    driver = webdriver.Chrome(service=service, options=options)
    return driver


In [ ]:
def normalize_name_for_college_id(player_name):
    # Turn "LeBron James" into "lebron-james-1" (this matches how the site names URLs)
    name = player_name.lower()
    # Replace spaces with hyphens
    name = re.sub(r'\s+', '-', name)
    # Player URLs end with "-1"
    college_id = f"{name}-1"
    return college_id


In [ ]:
def extract_table_from_html(soup, table_id):
    # Try to find the table directly in the HTML
    table = soup.find('table', id=table_id)
    if table:
        return pd.read_html(str(table))[0]

    # If it's hidden inside comments (the site does this), look there too
    comments = soup.find_all(string=lambda text: isinstance(text, Comment))
    for comment in comments:
        comment_soup = BeautifulSoup(comment, 'html.parser')
        table = comment_soup.find('table', id=table_id)
        if table:
            return pd.read_html(str(table))[0]

    # If still nothing found, return None
    return None


In [ ]:
def filter_stats_by_season(df, start_year=START_YEAR):
    # Skip if the dataframe is empty or missing the "Season" column
    if df is None or df.empty:
        return None
    if 'Season' not in df.columns:
        return None

    # Convert "2018-19" → 2018 (get the first four digits)
    def season_year(season_str):
        try:
            return int(season_str[:4])
        except:
            return None

    # Add a helper column to filter seasons, then drop it later
    df['SeasonYear'] = df['Season'].apply(season_year)
    filtered_df = df[df['SeasonYear'] >= start_year]
    if filtered_df.empty:
        return None

    # Remove the helper column before returning
    filtered_df = filtered_df.drop(columns=['SeasonYear'])
    return filtered_df


In [ ]:
def scrape_college_stats(driver, player_name):
    # Make the player’s college page link
    college_id = normalize_name_for_college_id(player_name)
    url = f"{CBB_BASE_URL}/{college_id}.html"

    # Open the page in the browser
    driver.get(url)

    # Small pause so the page fully loads
    time.sleep(3)

    # If the page doesn’t exist, skip it
    if "Page Not Found" in driver.page_source or "404" in driver.title:
        print(f"College page not found for {player_name} ({college_id})")
        return None, None

    # Parse the HTML to find the tables
    soup = BeautifulSoup(driver.page_source, 'html.parser')

    # Try to get per-game and advanced tables
    per_game_df = extract_table_from_html(soup, 'players_per_game')
    advanced_df = extract_table_from_html(soup, 'players_advanced')

    # Only keep data from START_YEAR and later
    per_game_df = filter_stats_by_season(per_game_df)
    advanced_df = filter_stats_by_season(advanced_df)

    return per_game_df, advanced_df


In [ ]:
def append_college_stats_to_csv(player_name, college_id, per_game_df, advanced_df, output_csv=COLLEGE_STATS_CSV):
    # If there's no data, just move on
    if per_game_df is None and advanced_df is None:
        print(f"Skipping {player_name} ({college_id}) because no college stats from {START_YEAR} onwards.")
        return

    # Add player info to the tables
    if per_game_df is not None:
        per_game_df = per_game_df.copy()
        per_game_df['PlayerName'] = player_name
        per_game_df['CollegeID'] = college_id
        print(f"Per-game columns for {player_name}: {per_game_df.columns.tolist()}")

    if advanced_df is not None:
        advanced_df = advanced_df.copy()
        advanced_df['PlayerName'] = player_name
        advanced_df['CollegeID'] = college_id
        print(f"Advanced columns for {player_name}: {advanced_df.columns.tolist()}")

    # Columns that both tables usually share
    merge_keys = ['Season', 'Team']

    # Merge both tables if both exist
    if per_game_df is not None and advanced_df is not None:
        merged_df = pd.merge(
            per_game_df,
            advanced_df,
            on=merge_keys,
            suffixes=('_per_game', '_adv'),
            how='outer
        )
        merged_df['PlayerName'] = player_name
        merged_df['CollegeID'] = college_id
    elif per_game_df is not None:
        merged_df = per_game_df
    else:
        merged_df = advanced_df

    # Write to CSV (make a new one or add to existing)
    if not os.path.isfile(output_csv):
        merged_df.to_csv(output_csv, index=False)
    else:
        merged_df.to_csv(output_csv, mode='a', header=False, index=False)

    print(f"Appended college stats for {player_name} ({college_id})")


In [ ]:
def main():
    # Start the browser
    driver = get_driver()

    # Make sure the player IDs file exists
    if not os.path.isfile(PLAYER_IDS_CSV):
        print(f"Player IDs CSV '{PLAYER_IDS_CSV}' not found. Please provide it with a 'PlayerName' column.")
        return

    # Load the list of players
    df_players = pd.read_csv(PLAYER_IDS_CSV)
    total_players = len(df_players)

    # Loop through every player and scrape their stats
    for idx, row in df_players.iterrows():
        player_name = row['PlayerName']
        college_id = normalize_name_for_college_id(player_name)
        print(f"Scraping college stats for {player_name} ({college_id}) [{idx+1}/{total_players}]")

        per_game_df, advanced_df = scrape_college_stats(driver, player_name)

        # Skip if no data found
        if per_game_df is None and advanced_df is None:
            print(f"No college stats found for {player_name}, skipping.")
            continue

        # Save their stats to the CSV
        append_college_stats_to_csv(player_name, college_id, per_game_df, advanced_df)

        # Small break between players
        time.sleep(3)

    # Close the browser when done
    driver.quit()


In [ ]:
# Run everything
if __name__ == "__main__":
    main()